# 02 — QSAR Fish Toxicity (LC50) Regression
Predict 96h fish LC50 (log scale) from 6 molecular descriptors. Data from the UCI Machine Learning Repository. This shows a common drug-like regression workflow.

In [ ]:
!pip -q install scikit-learn pandas numpy matplotlib

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
import warnings; warnings.filterwarnings('ignore')
RANDOM_STATE = 0

In [ ]:
# Download the dataset (semicolon-separated)
!wget -q "https://archive.ics.uci.edu/ml/machine-learning-databases/00504/qsar_fish_toxicity.csv" -O qsar_fish_toxicity.csv
cols = ["CIC0","SM1_Dz(Z)","GATS1i","NdsCH","NdssC","MLOGP","LC50"]
df = pd.read_csv("qsar_fish_toxicity.csv", sep=";", header=None, names=cols)
display(df.head())
print(df.shape)

In [ ]:
# Train/test split
X = df.drop(columns=["LC50"]).astype(float)
y = df["LC50"].astype(float)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)
print(X_train.shape, X_test.shape)

In [ ]:
# Toggle PCA on/off (Random Forests usually don't need PCA; try both)
USE_PCA = True
steps = [("scaler", StandardScaler())]
if USE_PCA:
    steps.append(("pca", PCA(n_components=5, random_state=RANDOM_STATE)))
steps.append(("rf", RandomForestRegressor(n_estimators=400, random_state=RANDOM_STATE, n_jobs=-1)))
pipe = Pipeline(steps)
pipe.fit(X_train, y_train)
pred = pipe.predict(X_test)
print("Test MAE (lower is better):", mean_absolute_error(y_test, pred))
print("Test R^2 (closer to 1 is better):", r2_score(y_test, pred))

In [ ]:
# 5-fold CV on the training set
cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_r2 = cross_val_score(pipe, X_train, y_train, cv=cv, scoring="r2", n_jobs=-1)
print("5-fold CV R^2 mean±sd:", cv_r2.mean(), "+/-", cv_r2.std())

In [ ]:
# Inference on a single new compound (placeholder values)
new_desc = pd.DataFrame([{ "CIC0": 1.2, "SM1_Dz(Z)": 0.5, "GATS1i": -0.1, "NdsCH": 2, "NdssC": 1, "MLOGP": 2.3 }])
print("Predicted log(LC50):", float(pipe.predict(new_desc)))

**Notes**
- The file uses **semicolon** separators.
- Try `USE_PCA = False` and compare scores.
- For interpretability, you can inspect `rf.feature_importances_`.